In [15]:
import spacy
from scispacy.linking import EntityLinker
from scispacy.linking_utils import KnowledgeBase
import os
import json
import re
from pathlib import Path
from pypdf import PdfReader as pr
#!pip install nltk
import pyobo
from nltk.tokenize import sent_tokenize
from pronto import Ontology

In [8]:
#os.chdir('..\\dissertation DLC content')
#linker = Ontology('OntoBiotope_BioNLP-OST-2019.obo')

path_to_scispacy_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\en_ner_bionlp13cg_md-0.5.4\\en_ner_bionlp13cg_md\\en_ner_bionlp13cg_md-0.5.4')
path_to_microbial_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\custom_microbial_ner')

scispacy_model = spacy.load(path_to_scispacy_model)
microbial_model = spacy.load(path_to_microbial_model)

#scispacy_model.add_pipe('scispacy_linker', config={'resolve_abbreviations': True, 'linker_name': 'chebi'})

c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\spacy\util.py:969: UserWarning: [W095] Model 'en_ner_bionlp13cg_md' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [ ]:
print(scispacy_model.pipe_names)

#linker = pyobo.from_obo_path('..\\dissertation DLC content\\chebi_lite.obo', version='1.2')
kb = KnowledgeBase('..\\dissertation DLC content\\chebi_lite.json')

linker = scispacy_model.add_pipe('entity_linker', after='ner')
linker.set_kb(kb)

onto = pyobo.fr('chebi_core')

#update this here to try and download a more relevant knowledge base
""" !python -m spacy_entity_linker "download_knowledge_base"

from spacy.kb import InMemoryLookupKB

def create_kb(vocab):
    kb = InMemoryLookupKB(vocab, entity_vector_length=128)
    return kb

linker = scispacy_model.add_pipe('entityLinker', after='ner') """

['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner']


TypeError: scispacy.linking_utils.Entity() argument after ** must be a mapping, not str

In [ ]:
with open('..\\dissertation\\list_files\\utilizations_list.json', 'r', encoding='utf-8') as f:
    utilizations = list(json.load(f))

basic_utilizations = ['fermentation', 'hydrolyze', 'hydrolyse', 'reduce', 'reduces', 'reduced', 'produce', 'production', 'produces', 'consume', 'consumes', 'consumption', 'consuming', 'producing', 'grow', 'growth', 'oxidise', 'oxidize', 'consumed', 'produced', 'convert', 'converts', 'converting', 'converted', 'conversion']

print('|'.join(basic_utilizations))
    
custom_labels = ['MICROBE_NAME', 'SIMPLE_CHEMICAL']

fermentation|hydrolyze|hydrolyse|reduce|reduces|reduced|produce|production|produces|consume|consumes|consumption|consuming|producing|grow|growth|oxidise|oxidize|consumed|produced|convert|converts|converting|converted|conversion


In [ ]:
for ind, file in enumerate(os.listdir(os.getcwd() + "\\fermentation_papers_preprocessed")[:5]):
    print(ind)
    filename = os.fsdecode(file)
    if filename.endswith('json'):
        with open(f"{os.getcwd()}\\fermentation_papers_preprocessed\\{filename}", 'r') as f:
            text = f.read()
    elif filename.endswith('pdf'):
        try:
            print('pdf')
            path = Path(os.getcwd() + "\\fermentation_papers\\" + filename)
            reader = pr(path)
            text = ""
            for page in reader.pages:
                text = text + page.extract_text()
        except:
            continue
    else:
        continue
    sentences = sent_tokenize(text)
    for sent in sentences:
        mic_ents = microbial_model(sent)
        mic_ents = [(mic.label_, mic.text, mic.start_char, mic.end_char) for mic in mic_ents.ents]
        sci_ents = scispacy_model(sent)
        sci_lents = sci_ents._.kb_ents
        linked_sci_ents = [lent.get_label() for lent in sci_lents]
        sci_tokens = [ent.text for ent in sci_ents]
        sci_ents = [(sci.label_, sci.text, sci.start_char, sci.end_char) for sci in sci_ents.ents if sci.text not in [text for l, text, s, e in mic_ents]]
        #simple chemicals should NOT be less than 3 characters long
        utils_found = re.finditer('|'.join(utilizations), str(sent).lower())
        utilization_ents = [('UTILIZATION', sent[util.start(0):util.end(0)].lower(), util.start(0), util.end(0)) for util in utils_found]
        ents_and_labels = [(label, text, start, end) for label, text, start, end in mic_ents + sci_ents if label in custom_labels and len(text)>=3] + utilization_ents
        ents_and_labels = sorted(ents_and_labels, key= lambda tup: tup[2])
        if ents_and_labels:
            print(sent)
            print(ents_and_labels)
            print(linked_sci_ents)
            print()

0


AttributeError: [E046] Can't retrieve unregistered extension attribute 'linkedEntities'. Did you forget to call the `set_extension` method?